# Projeto Final - Ecossistema de Dados


---


| Campo | Preencha |
|-------|----------|
| **Grupo** | |
| **Integrantes** | |
| **Dataset escolhido** | |
| **Fonte do dataset** | |
| **Data de entrega** | 17/04/2026 |


## Perguntas de negócio que este projeto responde

> Defina aqui as 5 perguntas de negócio que seu DW e Data Lake vão responder.  
> Estas perguntas guiarão toda a modelagem e as queries analíticas da Seção 6.

1. *(escreva aqui)*
2. *(escreva aqui)*
3. *(escreva aqui)*
4. *(escreva aqui)*
5. *(escreva aqui)*

---

## Decisão arquitetural

**Por que ETL para o Data Warehouse?**  
*(escreva aqui)*

**Por que ELT para o Data Lake?**  
*(escreva aqui)*

**Banco de dados escolhido para o DW e justificativa:**  
*(escreva aqui)*


---
## Estrutura do notebook

| Seção | Conteúdo |
|-------|----------|
| **Setup** | Instalação, imports, pastas |
| **Extract** | Leitura e exploração do dataset |
| **Raw/Staging** | Salvar bruto com metadados |
| **ETL → DW** | Qualidade → Dimensões → Fato → DuckDB |
| **ELT → Data Lake** | Bronze → Silver (SQL) → Gold (SQL) |
| **Monitoramento** | Log de pipeline + validação |
| **Analytics** | 5 queries OLAP respondendo as perguntas |
| **Conclusão** | Comparação ETL vs. ELT + recomendação |


---
# Seção 0: Setup


In [ ]:
# 0.1 - Instale as dependências
# Instale: duckdb, pandas, pyarrow
# Adicione outras que seu dataset exigir


In [ ]:
# 0.2 - Imports e configurações globais
import pandas as pd
import numpy as np
import duckdb
import json
import datetime
import os
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
# 0.3 - Criar estrutura de pastas do projeto
#
# Estrutura esperada:
#   Projeto Final/
#   ├── data/                  ← coloque o dataset aqui
#   ├── datalake/
#   │   ├── bronze/
#   │   ├── silver/
#   │   └── gold/
#   ├── logs/
#   └── dw_projeto.duckdb      ← criado na Seção 3

for pasta in ['data', 'datalake/bronze', 'datalake/silver', 'datalake/gold', 'logs']:
    Path(pasta).mkdir(parents=True, exist_ok=True)

print('Estrutura de pastas criada.')

### Função de Monitoramento

Esta função já está implementada. **Você deve chamá-la em toda etapa do pipeline.**  
Ela registra: nome da etapa, status, quantidade de registros antes e depois, e timestamp.

** Ajuste ao projeto, se for necessário


In [ ]:
# 0.4: Função de monitoramento do pipeline (NÃO altere esta célula)

PIPELINE_LOG = []

def log_etapa(etapa: str, status: str,
              qtd_antes: int = None, qtd_depois: int = None,
              obs: str = '') -> None:
    """
    Registra o resultado de uma etapa do pipeline.

    Parâmetros
    ----------
    etapa     : nome descritivo da etapa  (ex: 'ETL - Remover duplicatas')
    status    : 'OK' ou 'FALHA'
    qtd_antes : número de registros antes da operação
    qtd_depois: número de registros após a operação
    obs       : observação livre
    """
    removidos = (qtd_antes or 0) - (qtd_depois or 0) if qtd_antes and qtd_depois else None
    entrada = {
        'etapa'     : etapa,
        'status'    : status,
        'qtd_antes' : qtd_antes,
        'qtd_depois': qtd_depois,
        'removidos' : removidos,
        'timestamp' : datetime.datetime.now().isoformat(),
        'obs'       : obs
    }
    PIPELINE_LOG.append(entrada)
    qtd_str = f'{qtd_depois:,}' if qtd_depois is not None else '-'
    rem_str = f'  ({removidos:+,} registros)' if removidos else ''
    print(f"[{status:^5}]  {etapa:<45}  {qtd_str} registros{rem_str}")


def exibir_log() -> pd.DataFrame:
    """Exibe o log completo do pipeline como DataFrame."""
    if not PIPELINE_LOG:
        print('Nenhuma etapa registrada ainda.')
        return None
    return pd.DataFrame(PIPELINE_LOG)


def salvar_log(caminho: str = 'logs/pipeline_log.json') -> None:
    """Salva o log em JSON para auditoria."""
    with open(caminho, 'w', encoding='utf-8') as f:
        json.dump(PIPELINE_LOG, f, ensure_ascii=False, indent=2)
    print(f'Log salvo em: {caminho}')


print('Função log_etapa disponível.')

---
# Seção 1: Extract

> **Objetivo:** carregar o dataset e entender o que você tem antes de qualquer transformação.


In [ ]:
# 1.1 - Leia o dataset
#
# Dica: se houver problema de encoding, tente encoding='latin-1' ou encoding='windows-1252'
# Dica: se o separador for ponto-e-vírgula, use sep=';'
#
# Ao final, registre no log:
#   log_etapa('Extract - Leitura do dataset', 'OK', qtd_depois=len(df))


In [ ]:
# 1.2 - Exploração inicial
#
# Exiba:
#   a) Shape (linhas × colunas)
#   b) Tipos de dados de cada coluna (dtypes)
#   c) Contagem de nulos por coluna
#   d) Estatísticas descritivas das colunas numéricas
#   e) As 5 primeiras linhas


In [ ]:
# 1.3 - Diagnóstico de qualidade
#
# Investigue e documente:
#   a) Quantidade de linhas duplicadas
#   b) Colunas com mais de 10% de nulos
#   c) Colunas de data que estão como string
#   d) Valores inválidos em colunas numéricas (negativos onde não deveria)
#   e) Inconsistências de padronização (ex: estados em maiúsculo e minúsculo)
#
# Este diagnóstico guiará o tratamento na Seção 3


---
# Seção 2: Staging

> **Regra fundamental:** a Staging recebe o dado **bruto**, exatamente como veio da fonte.  
> Nenhuma transformação é aplicada aqui, apenas metadados de ingestão são adicionados.
>
> Por que isso importa? Se o pipeline falhar depois, você pode reprocessar a partir do staging  
> sem precisar voltar à fonte original.


In [ ]:
# 2.1 - Salvar o dataset bruto como Parquet no staging
#
# Antes de salvar, adicione as colunas de metadados:
#   _source_file  : nome do arquivo de origem (string)
#   _ingested_at  : timestamp da ingestão (pd.Timestamp.now())
#   _batch_id     : identificador da execução, use datetime.date.today().isoformat()
#
# Salvar em: datalake/bronze/staging_raw.parquet
#
# Ao final:
#   log_etapa('Staging - Salvar bruto', 'OK', qtd_antes=len(df), qtd_depois=len(df_staging))


In [ ]:
# 2.2: Verificar o arquivo de staging
#
# Leia o Parquet de volta e confirme:
#   a) O número de linhas bate com o dataset original
#   b) As colunas de metadados estão presentes
#   c) Nenhuma transformação foi aplicada (os dados são idênticos ao CSV)


---
# Seção 3: ETL → Data Warehouse

## Fluxo
```
Staging  ──►  TRANSFORM (Python)  ──►  LOAD  ──►  dw_projeto.duckdb
```

> Toda a lógica de negócio acontece em Python, **antes** de qualquer dado ir para o banco.  
> O DW recebe apenas DataFrames já limpos e modelados.

## Seu modelo Star Schema

*(Substitua o diagrama abaixo pelo modelo que seu grupo projetou)*

```
                dim_data
                    │
 dim_X  ────  fato_principal  ────  dim_Y
                    │
                dim_Z
```

| Tabela | Colunas | Origem |
|--------|---------|--------|
| `dim_data` | sk_data, data, ano, trimestre, mes, dia_semana, is_weekend | coluna de data |
| `dim_...` | sk_..., ... | ... |
| `fato_...` | sk_fato, sk_data, sk_..., métricas | todas |


## 3.1: EXTRACT: Tratamento de Qualidade

> Trabalhe sobre o DataFrame lido do staging (não do CSV original).  
> Documente cada decisão: o que foi removido e por quê.


In [ ]:
# 3.1a - Carregar o staging para trabalhar
#
# Leia datalake/bronze/staging_raw.parquet
# Trabalhe sempre a partir daqui - não do CSV original


In [ ]:
# 3.1b - Remover duplicatas
#
# Identifique as colunas que definem uma linha única
# Exiba a quantidade antes e depois
#
# log_etapa('ETL - Remover duplicatas', 'OK', qtd_antes=..., qtd_depois=...)


In [ ]:
# 3.1c - Converter tipos de dados
#
# Converta colunas de data para datetime
# Converta colunas numéricas que estejam como string
#
# log_etapa('ETL - Converter tipos', 'OK', qtd_depois=len(df_clean))


In [ ]:
# 3.1d - Tratar nulos e valores inválidos
#
# Para cada coluna com problema, decida e justifique:
#   - Remover a linha?
#   - Preencher com valor padrão / mediana / moda?
#   - Manter como está e tratar na query?
#
# log_etapa('ETL - Tratar nulos', 'OK', qtd_antes=..., qtd_depois=...)


In [ ]:
# 3.1e - Padronizações (se necessário)
#
# Exemplos: estados em minúsculo → maiúsculo, strip de espaços,
# categorias com nomes inconsistentes, etc.
#
# log_etapa('ETL - Padronizações', 'OK', qtd_depois=len(df_clean))


## 3.2: TRANSFORM: Construção das Dimensões

> Para cada dimensão: extraia as colunas relevantes, elimine duplicatas  
> e adicione uma **Surrogate Key (SK)** sequencial começando em 1.


In [ ]:
# 3.2a - Construir dim_data
#
# Colunas obrigatórias: sk_data, data, ano, trimestre, mes, nome_mes,
#                       dia_semana, nome_dia, is_weekend
#
# Dica: use pd.to_datetime e os acessores .dt.year, .dt.quarter, etc.
#
# log_etapa('ETL - Construir dim_data', 'OK', qtd_depois=len(dim_data))


In [ ]:
# 3.2b - Construir segunda dimensão (defina o nome)
#
# Exemplo: dim_cliente, dim_produto, dim_local, dim_vendedor...
#
# Extraia as colunas relevantes do df_clean
# Elimine duplicatas (.drop_duplicates())
# Gere SK sequencial (.reset_index(drop=True); dim['sk_X'] = dim.index + 1)
#
# log_etapa('ETL - Construir dim_...', 'OK', qtd_depois=len(dim_X))


In [ ]:
# 3.2c - Construir terceira dimensão (se houver)
#
# log_etapa('ETL - Construir dim_...', 'OK', qtd_depois=len(dim_Y))


In [ ]:
# 3.2d - Construir demais dimensões (adicione células conforme necessário)


In [ ]:
# 3.2e - Construir a tabela fato
#
# log_etapa('ETL - Construir fato', 'OK', qtd_depois=len(fato))


## 3.3: LOAD: Criar o DW e carregar as tabelas

> **Ordem obrigatória:** dimensões primeiro, fato por último.  
> A fato contém FKs que referenciam as dimensões, carregar na ordem errada quebra a integridade.


In [ ]:
# 3.3a - Criar o banco DuckDB e o DDL de cada tabela
#
# 1. Conecte ao banco: conn = duckdb.connect('dw_projeto.duckdb')
# 2. Escreva o CREATE TABLE para cada dimensão
#    - Defina tipos de dados explicitamente
#    - Defina a PK (PRIMARY KEY) na SK
# 3. Escreva o CREATE TABLE para a fato
#    - Defina FOREIGN KEY para cada dimensão
#
# Exemplo de DDL:
# conn.execute("""
#     CREATE TABLE IF NOT EXISTS dim_data (
#         sk_data     INTEGER PRIMARY KEY,
#         data        DATE,
#         ano         INTEGER,
#         trimestre   INTEGER,
#         mes         INTEGER,
#         nome_mes    VARCHAR,
#         dia_semana  INTEGER,
#         nome_dia    VARCHAR,
#         is_weekend  BOOLEAN
#     )
# """)


In [ ]:
# 3.3b - Carregar os DataFrames no DW
#
# Use: conn.execute('INSERT INTO tabela SELECT * FROM df_python')
# ou:  conn.register('df_view', df_python)
#       conn.execute('INSERT INTO tabela SELECT * FROM df_view')
#
# Ordem: dimensões primeiro, fato por último
#
# log_etapa('DW - Carregar dim_data',  'OK', qtd_depois=...)
# log_etapa('DW - Carregar dim_...',   'OK', qtd_depois=...)
# log_etapa('DW - Carregar fato_...',  'OK', qtd_depois=...)


In [ ]:
# 3.3c - Validar o DW
#
# Para cada tabela, execute: SELECT COUNT(*) FROM tabela
# Verifique:
#   a) A contagem bate com os DataFrames?
#   b) A soma das métricas da fato bate com o total do dataset original?
#   c) Existem SKs nulas na fato?
#
# log_etapa('DW - Validação pós-carga', 'OK', qtd_depois=total_fato)


---
# Seção 4: ELT → Data Lake (Arquitetura Medallion)

## Fluxo
```
Bronze ──► Silver ──► Gold
```

> Os mesmos dados da Seção 3, mas desta vez o dado é carregado bruto primeiro.  
> As transformações acontecem em SQL, dentro do Data Lake.  
> Python serve apenas para orquestrar, toda lógica está nas queries.


## 4.1: Camada Bronze (dado bruto)

> Bronze é um **espelho fiel da fonte**.  
> Nenhum dado é alterado, removido ou filtrado.  
> É idêntico ao staging, a diferença é que agora está na estrutura do Data Lake.


In [ ]:
# 4.1 - Salvar o dado bruto na camada Bronze
#
# Leia o staging_raw.parquet e salve em datalake/bronze/bronze_dataset.parquet
# O arquivo Bronze deve ser idêntico ao staging
#
# Confirme: exiba o número de linhas e colunas
#
# log_etapa('ELT - Bronze (LOAD)', 'OK', qtd_depois=...)


## 4.2: Camada Silver (limpeza com SQL)

> Na Silver, use **apenas SQL** sobre o arquivo Bronze.  
> O DuckDB lê Parquet diretamente: `read_parquet('datalake/bronze/bronze_dataset.parquet')`  
> Para salvar: `COPY (SELECT ...) TO 'datalake/silver/silver.parquet' (FORMAT PARQUET)`


In [ ]:
# 4.2a - Conectar ao DuckDB para as transformações SQL
#
# Use uma conexão separada (ou a mesma do DW - DuckDB suporta múltiplas)
# conn_lake = duckdb.connect()  # in-memory é suficiente para o Data Lake


In [ ]:
# 4.2b - Construir a Silver com SQL
#
# Escreva uma única query que:
#   1. Leia o Bronze: FROM read_parquet('datalake/bronze/bronze_dataset.parquet')
#   2. Renomeie colunas com espaços usando aliases ("Coluna X" AS coluna_x)
#   3. Converta colunas de data: STRPTIME(coluna, '%m/%d/%Y') ou TRY_CAST(col AS DATE)
#   4. Remova duplicatas: QUALIFY ROW_NUMBER() OVER (...) = 1
#   5. Filtre registros inválidos (métricas negativas, datas nulas...)
#   6. Adicione colunas calculadas relevantes para o negócio
#
# Salve em: datalake/silver/silver_dataset.parquet
#
# log_etapa('ELT - Silver (TRANSFORM SQL)', 'OK', qtd_antes=..., qtd_depois=...)


In [ ]:
# 4.2c - Verificar a Silver
#
# Leia o Parquet e exiba: shape, schema (.dtypes) e 3 primeiras linhas
# Confirme que as transformações foram aplicadas corretamente


## 4.3: Camada Gold (agregações orientadas ao negócio)

> A Gold responde diretamente às perguntas de negócio definidas no início do notebook.  
> Crie uma tabela Gold para cada tema analítico que seu grupo definiu.


In [ ]:
# 4.3a - Tabela Gold 1 (defina o nome e o tema)
#
# Exemplos:
#   gold_por_categoria   - vendas e lucro agregados por categoria
#   gold_mensal          - evolução temporal de métricas
#   gold_por_regiao      - performance geográfica
#   gold_por_segmento    - análise por segmento de cliente
#
# Use SQL sobre a Silver:
#   FROM read_parquet('datalake/silver/silver_dataset.parquet')
#
# Salve em: datalake/gold/gold_tema1.parquet
#
# log_etapa('ELT - Gold (tema1)', 'OK', qtd_depois=...)


In [ ]:
# 4.3b - Tabela Gold 2
#
# log_etapa('ELT - Gold (tema2)', 'OK', qtd_depois=...)


In [ ]:
# 4.3c - Tabela Gold 3
#
# log_etapa('ELT - Gold (tema3)', 'OK', qtd_depois=...)


---
# Seção 5: Monitoramento

> Exibir e salvar o log completo do pipeline.  
> Se todas as etapas anteriores chamaram `log_etapa`, o relatório será gerado automaticamente.


In [ ]:
# 5.1 - Exibir o log completo do pipeline
df_log = exibir_log()
display(df_log)

In [ ]:
# 5.2 - Resumo de qualidade
#
# Calcule e exiba:
#   a) Total de etapas executadas
#   b) Etapas com status OK vs. FALHA
#   c) Total de registros removidos em todo o pipeline (soma da coluna 'removidos')
#   d) Percentual de dados perdidos em relação ao dataset original
#
# Exemplo:
# print(f"Etapas OK    : {df_log[df_log.status=='OK'].shape[0]}")
# print(f"Etapas FALHA : {df_log[df_log.status=='FALHA'].shape[0]}")


In [ ]:
# 5.3 - Salvar o log em JSON
salvar_log('logs/pipeline_log.json')

In [ ]:
# 5.4 - Validação cruzada: DW vs. Data Lake
#
# Compare os totais entre o DW e a camada Gold:
#   a) A soma da métrica principal na fato do DW bate com a Gold?
#   b) O número de registros da Silver bate com a fato do DW (após tratamentos)?
#
# Se houver divergência, investigue e documente


---
# Seção 6: Analytics (OLAP)

> Execute as 5 queries que respondem às perguntas de negócio definidas no início.  
> Cada query deve ser executada **no DW** (com JOIN nas dimensões) **e também na camada Gold** (comparando os resultados).


In [ ]:
# Query 1 - (copie aqui a pergunta de negócio 1)
#
# Responder no DW (use JOIN com pelo menos uma dimensão):


In [ ]:
# Query 1 - mesma pergunta, respondida na camada Gold:
# FROM read_parquet('datalake/gold/gold_tema_X.parquet')


In [ ]:
# Query 2 - (copie aqui a pergunta de negócio 2)
#
# No DW:


In [ ]:
# Query 2 - na camada Gold:


In [ ]:
# Query 3 - (copie aqui a pergunta de negócio 3)
#
# No DW:


In [ ]:
# Query 3 - na camada Gold:


In [ ]:
# Query 4 - (copie aqui a pergunta de negócio 4)
#
# No DW:


In [ ]:
# Query 4 - na camada Gold:


In [ ]:
# Query 5 - (copie aqui a pergunta de negócio 5)
#
# No DW:


In [ ]:
# Query 5 - na camada Gold:


---
# Seção 7: Conclusão

Responda as questões abaixo com base na experiência de implementar este projeto.

---

## 7.1 - ETL vs. ELT no contexto do seu dataset

**a) Qual abordagem foi mais trabalhosa de implementar? Por quê?**  
*(resposta do grupo)*

**b) As queries no DW (com JOIN nas dimensões) e na camada Gold retornaram os mesmos resultados?  
Se houve divergência, o que a causou?**  
*(resposta do grupo)*

**c) Se os dados da fonte forem atualizados (novos registros chegarem amanhã),  
qual pipeline seria mais fácil de reexecutar - ETL ou ELT? Justifique.**  
*(resposta do grupo)*

---

## 7.2 - Recomendação arquitetural

**Se este projeto fosse colocado em produção para uma empresa real,  
qual arquitetura o grupo recomendaria?**

Considere:
- Volume dos dados e frequência de atualização
- Perfil dos usuários (analistas de negócio, cientistas de dados, engenheiros)
- Necessidade de auditoria e rastreabilidade
- Custo de manutenção do pipeline

*(resposta do grupo - mínimo 5 linhas)*

---

## 7.3 - O que vocês fariam diferente

**Cite 2 decisões que tomariam de forma diferente se recomeçassem o projeto hoje.**

1. *(resposta)*
2. *(resposta)*


In [ ]:
# Encerramento - fechar conexões e exibir resumo final

try:
    conn.close()
except:
    pass

try:
    conn_lake.close()
except:
    pass

print('\n' + '='*60)
print('  PROJETO FINAL - RESUMO DE EXECUÇÃO')
print('='*60)

df_log = exibir_log()
if df_log is not None:
    ok    = df_log[df_log['status'] == 'OK'].shape[0]
    falha = df_log[df_log['status'] == 'FALHA'].shape[0]
    print(f'Etapas executadas : {len(df_log)}')
    print(f'OK                : {ok}')
    print(f'FALHA             : {falha}')
    removidos = df_log['removidos'].sum()
    print(f'Registros removidos no pipeline: {removidos:,.0f}')

print('='*60)
salvar_log('logs/pipeline_log.json')